In [ ]:
# !pip install arxiv

In [ ]:
import os 
import arxiv
import requests

os.makedirs("Data/papers" , exist_ok = True)
#create the client
client = arxiv.Client()

#serach creation
search = arxiv.Search(
    query = "retrivel augmented generation" , 
    max_results = 30 , 
    sort_by = arxiv.SortCriterion.SubmittedDate
)
# we get client result 
results = client.results(search)

for i , result in enumerate(results):
    # Paper file name create 
    paper_id = result.get_short_id().replace("/" , "_")
    filename = f"Data/papers/{i+1:02d}_{paper_id}.pdf"

    print(f"{i+1}. Downloading: {result.title[:70]}...")

    response = requests.get(result.pdf_url)

    with open(filename , "wb") as f:
        f.write(response.content)
    
print("Done! 30PDFs Downloaded")

In [ ]:
# pip install -U langchain-text-splitters

In [ ]:
pip install pymupdf

In [ ]:
import fitz # to read and load the pdf content images 
import os 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# first we get pdf
pdf_files = os.listdir("Data/papers")
first_pdf = f"Data/papers/{pdf_files[0]}"
print(f"Paper: {first_pdf}")

#Then extract the content of pdf
doc= fitz.open(first_pdf)
full_text = ""
for page in doc:
    full_text += page.get_text() +"\n"

print(f"Total character in paper: {len(full_text)}")
print(f"Total Pages: {len(doc)}")
print("\n-- 500 charactesr --")
print(full_text[ : 500])
# small_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# texts = small_splitter.split_text(documen)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=50)
chunk_small = small_splitter.split_text(full_text)

large_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100)
chunk_large = large_splitter.split_text(full_text)

print(f"chunk size 500 -> Total chunks {len(chunk_small)}")
print(f"chunk size 100 -> Total Chunks {len(chunk_large)}")

print("-- 500 chunk NO. 5--")
print(chunk_small[5])

print("--100 chunk NO. 5 --")
print(chunk_large[5])

In [ ]:
# !pip install sentence-transformers scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np 

#1 Model loading 
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences= [
    "Posterior Matching Distillation is used for image generation",
    "PMD is used to generate images from text",
    "The cat is sitting on the table"
]

# sentence can convert into the vectors  
embeddings = model.encode(sentences)

print(f"Total sentences: {len(sentences)}")
print(f"sentence vector size: {embeddings.shape[1]} dimensions")
print(f"\n the first sentence vector size ( first 10 numbers) : ")
print(embeddings[0][ : 10])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print("Embedding 0 ke pehle 3 numbers:", embeddings[0][:3])
print("Embedding 1 ke pehle 3 numbers:", embeddings[1][:3])
print("Embedding 2 ke pehle 3 numbers:", embeddings[2][:3])

sim_1_2 = cosine_similarity([embeddings[0],embeddings[1]])[0][0]
sim_1_3= cosine_similarity([embeddings[0], embeddings[2]])[0][0]

print(f"Similarity (Sentence 1 vs 2) - Same Meaning: {sim_1_2:.4f}")

print(f"Similarity (Sentence 1 vs 3) - Different Meaning: {sim_1_3:.4f}")

In [ ]:
import numpy as np

print("=== FRESH CHECK ===")
print("Shape:", embeddings.shape)
print("Kya 0 aur 1 same hai?", np.allclose(embeddings[0], embeddings[1]))
print("Kya 0 aur 2 same hai?", np.allclose(embeddings[0], embeddings[2]))

# Difference dekho
diff = np.linalg.norm(embeddings[0] - embeddings[1])
print("Difference 0 vs 1 (0 hoga to same hai):", diff)

# Manual cosine - ekdum fresh
a = embeddings[0]
b = embeddings[1]
c = embeddings[2]

dot_1_2 = np.dot(a, b)
norm = np.linalg.norm(a) * np.linalg.norm(b)
print("Manual Cosine 1 vs 2:", dot_1_2 / norm)

dot_1_3 = np.dot(a, c)
norm2 = np.linalg.norm(a) * np.linalg.norm(c)
print("Manual Cosine 1 vs 3:", dot_1_3 / norm2)

In [ ]:
import fitz
import os 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# model loading for  30 pdf 
print("model Loading ..")
model  = SentenceTransformer("all-MiniLM-L6-v2", device = "cuda")

# chunking the documents  
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

all_chunks = []
all_metadatas = [] # metadata means which pdf data will comes 

# initialize the loops for 30 pdf '
pdf_files = os.listdir("Data/papers")
print(f"Total PDFS mile : {len(pdf_files)}")

for pdf_file in pdf_files:
    pdf_path = f"Data/papers/{pdf_file}"

    # Extract the text into the pdf
    doc= fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text() + "\n"

    # Text to chunks breakdownns
    chunks = splitter.split_text(full_text)
    print(f"{pdf_file} -> Page : {len(doc)} , total chunks: {len(chunks)}")

    for chunk in chunks:
        all_chunks.append(chunk)
        all_metadatas.append({"source" : pdf_file})

print(f"\n == Final Result ==")
print(f"Total Chunks in 30 pdf : {len(all_chunks)}")

# all chunks we convert into embedding at one time 
print("Embeddings are create plsease wait..")
all_embeddings = model.encode(all_chunks , show_progress_bar = True)

print(f"Tota Embeddings : {all_embeddings.shape}")
print(f"The Size of Embedding : {all_embeddings.shape[1]} dimensions")

In [ ]:
# !pip install chromadb

In [ ]:
import chromadb

# we create the client for chroma 
client = chromadb.PersistentClient(path = "./chroma_db")

# collection are we create for exmaple database are consisting a table 

collection = client.get_or_create_collection(name="paper_30")

#3  add 3756 chunks in chroma db and also asign the ids for chromadb
if collection.count() == 0:
    ids = [f"chunk_{i}" for i in range(len(all_chunks))]
    
    #Embeddiings are convert into list 
    embeddings_list = all_embeddings.tolist()
    
    print("Chroma  ADDD ")
    collection.add(
        documents = all_chunks,
        embeddings= embeddings_list,
        metadatas =all_metadatas,
        ids = ids
    )
    print(f"Done! Total chunks: {collection.count()}")
else:
    print("already added we cannot load again")

# print(f"Done! Chroma are loded the total chunks: {collecion.count()}")

In [ ]:
import torch
print("CUDA Available hai?", torch.cuda.is_available())
print("GPU ka naam:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU nahi mila")

In [ ]:
# pip install groq

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq

In [ ]:
from dotenv import load_dotenv
load_dotenv()

model = SentenceTransformer("all-MiniLM-L6-v2",device = "cpu")

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name = "paper_30")

# we joined the groq api key 
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

#user Question
query = "What is the difference between trajectory-based and distribution-based distillation?"
query_embedding = model.encode([query]).tolist()
results = collection.query(query_embeddings = query_embedding , n_results = 5)

context = "\n\n".join(results['documents'][0])
print("Retrived Context are finding out ")

# Generation -Groq
prompt = f""" You are a research assistant. Answer ONLY in English and ONLY from the given context.
If answer is not in context, say "I don't know from these papers".

            Context :{context}

            Answer with citaion of source paper :
            """
response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages = [{"role": "user" , "content": prompt}]
)

print("\n == FINAL RAG ANSWER ==")
print(response.choices[0].message.content)
print("\n == SOURCES == ")
for m in results["metadatas"][0]:
    print(m["source"])

In [ ]:
!pip install rank_bm25

# WE are implement the BM25 IMplementaion

In [ ]:
from rank_bm25 import BM25Okapi

# we break the chunks into the words -> because BM25 needs words 
tokenizes_chunks = [chunk.lower().split() for chunk in all_chunks]

#BM25 indexing are we create 
bm25 = BM25Okapi(tokenizes_chunks)
print("BM25 Index created ")

# query 
query = "What is PMD"
tokenized_query = query.lower().split()

# we fetech the top 3 chunks using the BM25
bm25_scores = bm25.get_scores(tokenized_query)

# top 3 chunks get index
import numpy as np 
top_3_idx = np.argsort(bm25_scores)[::-1][:3]

print(f"query: {query}\n")
for i , idx in enumerate(top_3_idx):
    print(f"--- BM25 Result {i+1} | Source: {all_metadatas[idx]['source']} | Score: {bm25_scores[idx]:.2f} ---")
    print(all_chunks[idx][:400])
    print()


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np 
import chromadb
from sentence_transformers import SentenceTransformer
import re

In [ ]:
# Loading the model
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="paper_30")

# Query
query = "What is PMD?"

# === FIX: Ek hi cleaning function dono ke liye ===
def clean_text(text):
    return re.sub(r'[^\w\s]', '', text.lower()).split()

# Part A: Dense Search
query_embedding = model.encode([query]).tolist()
dense_result = collection.query(query_embeddings=query_embedding, n_results=5)
dense_docs = dense_result['documents'][0]

# Part B: BM25 - Ab chunks bhi saaf hai, query bhi saaf hai
tokenized_chunks = [clean_text(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)

# Stopwords hatao - sirf "pmd" bachega
stopwords = {"what", "is", "the", "a", "an", "for", "of", "how", "why", "does"}
query_tokens = [w for w in clean_text(query) if w not in stopwords]
print("BM25 ke liye final query:", query_tokens)

bm25_scores = bm25.get_scores(query_tokens)
bm25_idx = np.argsort(bm25_scores)[::-1][:5]
bm25_docs = [all_chunks[i] for i in bm25_idx]

print("Dense Top 1:", dense_docs[0][:100])
print("BM25 Top 1:", bm25_docs[0][:100])

# RRF Fusion - Ye part same hai
k = 60
rrf_scores = {}
for rank, doc in enumerate(dense_docs):
    rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (k + rank + 1)
for rank, doc in enumerate(bm25_docs):
    rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (k + rank + 1)

final_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:3]
for i, (doc, score) in enumerate(final_docs):
    print(f"\n--- Hybrid Result {i+1} | RRF Score: {score:.4f} ---")
    print(doc[:500])

In [ ]:
# !pip install sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

# 1. Reranker model load karo - Ye chhota model hai, CPU pe bhi chal jayega
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device="cpu")
print("Reranker load ho gaya!")

# 2. Tumhare Hybrid ke Top 5 chunks lo (jo pichle step me mile the)
# Yaha main example ke liye final_docs use kar raha hu
query = "What is PMD?"

# Hybrid se mile hue docs ko Reranker ko do - Pair banao [Query, Chunk]
pairs = [[query, doc] for doc, score in final_docs]

# 3. Reranker se naya score nikalo
rerank_scores = reranker.predict(pairs)

print(f"Query: {query}\n")
for i, (doc, old_score) in enumerate(final_docs):
    print(f"Chunk {i+1} | Old RRF Score: {old_score:.4f} | New Rerank Score: {rerank_scores[i]:.4f}")
    print(doc[:300])
    print("---")

# 4. Rerank Score se dobara sort karo
reranked = sorted(zip(final_docs, rerank_scores), key=lambda x: x[1], reverse=True)

print("\n=== FINAL RERANKED TOP 1 ===")
print(reranked[0][0][0][:500])

# model Evaluation

In [ ]:
# 10 Test Questions - Inke answer tumhare 30 papers me hai
test_questions = [
    {"q": "What is Posterior Matching Distillation PMD?", "expected_keyword": "posterior"},
    {"q": "What is Distribution Matching Distillation DMD?", "expected_keyword": "distribution"},
    {"q": "What is the difference between trajectory and distribution based distillation?", "expected_keyword": "trajectory"},
    {"q": "What is Consistency Distillation?", "expected_keyword": "consistency"},
    {"q": "What is Flow Map Distillation?", "expected_keyword": "flow"},
    {"q": "What is the role of teacher model in distillation?", "expected_keyword": "teacher"},
    {"q": "What is mode collapse in diffusion models?", "expected_keyword": "mode"},
    {"q": "What is KL divergence used for in distillation?", "expected_keyword": "KL"},
    {"q": "How many steps does PMD need for image generation?", "expected_keyword": "four"},
    {"q": "What is the advantage of PMD over DMD?", "expected_keyword": "diversity"},
]

print(f"Total Test Questions: {len(test_questions)}")

In [ ]:
import time
import re
import numpy as np
from rank_bm25 import BM25Okapi

# Pehle BM25 ek baar bana lo - Har baar mat banao
def clean_text(text):
    return re.sub(r'[^\w\s]', '', text.lower()).split()

tokenized_chunks = [clean_text(chunk) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_chunks)
stopwords = {"what", "is", "the", "a", "an", "for", "of", "how", "why", "does", "in", "to"}

# Reranker bhi ek baar load karo
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device="cpu")

def evaluate_baseline(question):
    # Sirf Dense Search
    q_emb = model.encode([question]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=5)
    return res['documents'][0]

def evaluate_advanced(question):
    # Step 1: Dense Top 5
    q_emb = model.encode([question]).tolist()
    dense_docs = collection.query(query_embeddings=q_emb, n_results=5)['documents'][0]
    
    # Step 2: BM25 Top 5
    query_tokens = [w for w in clean_text(question) if w not in stopwords]
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_idx = np.argsort(bm25_scores)[::-1][:5]
    bm25_docs = [all_chunks[i] for i in bm25_idx]
    
    # Step 3: RRF Fusion
    k = 60
    rrf_scores = {}
    for rank, doc in enumerate(dense_docs):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (k + rank + 1)
    for rank, doc in enumerate(bm25_docs):
        rrf_scores[doc] = rrf_scores.get(doc, 0) + 1 / (k + rank + 1)
    
    # Top 10 RRF wale nikalo (5 Dense + 5 BM25 me se)
    hybrid_docs = [doc for doc, score in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:10]]
    
    # Step 4: Rerank Top 10 ko
    pairs = [[question, doc] for doc in hybrid_docs]
    rerank_scores = reranker.predict(pairs)
    reranked = sorted(zip(hybrid_docs, rerank_scores), key=lambda x: x[1], reverse=True)
    
    # Final Top 5 return karo
    final_docs = [doc for doc, score in reranked[:5]]
    return final_docs

# Test Questions
test_questions = [
    {"q": "What is Posterior Matching Distillation PMD?", "expected_keyword": "posterior"},
    {"q": "What is Distribution Matching Distillation DMD?", "expected_keyword": "distribution"},
    {"q": "What is the difference between trajectory and distribution based distillation?", "expected_keyword": "trajectory"},
    {"q": "What is Consistency Distillation?", "expected_keyword": "consistency"},
    {"q": "What is Flow Map Distillation?", "expected_keyword": "flow"},
    {"q": "What is the role of teacher model in distillation?", "expected_keyword": "teacher"},
    {"q": "What is mode collapse in diffusion models?", "expected_keyword": "mode"},
    {"q": "What is KL divergence used for in distillation?", "expected_keyword": "KL"},
    {"q": "How many steps does PMD need for image generation?", "expected_keyword": "four"},
    {"q": "What is the advantage of PMD over DMD?", "expected_keyword": "diversity"},
]

# Ab Evaluation chalao
total_baseline_hit = 0
total_advanced_hit = 0

for item in test_questions:
    q = item["q"]
    keyword = item["expected_keyword"].lower()
    
    # Baseline
    start = time.time()
    baseline_docs = evaluate_baseline(q)
    baseline_time = time.time() - start
    baseline_hit = sum(1 for doc in baseline_docs if keyword in doc.lower())
    
    # Advanced
    start = time.time()
    advanced_docs = evaluate_advanced(q)
    advanced_time = time.time() - start
    advanced_hit = sum(1 for doc in advanced_docs if keyword in doc.lower())
    
    total_baseline_hit += baseline_hit
    total_advanced_hit += advanced_hit
    
    print(f"Q: {q}")
    print(f"  Baseline (Dense)   - Hit: {baseline_hit}/5 | Time: {baseline_time:.2f}s")
    print(f"  Advanced (Hybrid+Rerank) - Hit: {advanced_hit}/5 | Time: {advanced_time:.2f}s")
    print()

print(f"=== FINAL RESULT ===")
print(f"Baseline Total Hit: {total_baseline_hit} / 50")
print(f"Advanced Total Hit: {total_advanced_hit} / 50")
print(f"Improvement: {total_advanced_hit - total_baseline_hit} chunks zyada sahi mile")

In [ ]:
# 1. Ye function - Answer + Citation banayega
def get_answer_with_citation(query, history_text=""):
    docs = evaluate_advanced(query)  # Tumhara Phase 4 wala function - Already hai

    # Citations - Metadata se
    citations = []
    for doc in docs[:2]:
        try:
            idx = all_chunks.index(doc)
            meta = all_metadatas[idx]
            citations.append(f"{meta.get('source', 'Paper')} - Page {meta.get('page', '?')}")
        except:
            citations.append("Source: PaperMind DB")

    context = "\n\n".join(docs)
    prompt = f"""Chat History: {history_text}\n\nContext: {context}\n\nQuestion: {query}\nAnswer in simple words."""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content, citations

# 2. Tumhara wala Chat function - Isme thoda sa History add kiya
chat_history = []
answer_cache = {}
chat_history = []  # Stateful - Last 2 questions
answer_cache = {}  # Cache - Hit / Miss

def chat(query):
    # Step 1: Cache Check - System Design: Latency + Cost bachao
    if query.lower() in answer_cache:
        return answer_cache[query.lower()]  # Cache Hit: 0.001s
    
    # Step 2: History - System Design: Sliding Window
    history = chat_history[-2:]  # Sirf last 2
    
    # Step 3: Retrieval + Rerank - System Design: Accuracy
    docs = evaluate_advanced(query)
    
    # Step 4: LLM + Citations - System Design: Trust
    answer, citations = get_answer_with_citation(query)
    
    # Step 5: Save for next time
    answer_cache[query.lower()] = answer
    chat_history.append((query, answer))
    return answer

In [ ]:
chat("What is PMD?")  # Pehli baar - RAG chalega
chat("What is PMD?")  # Dusri baar - Cache se aayega

# MULTIMODLE RAG

In [ ]:
import fitz  # PyMuPDF
import os

# 1. Folder banao
os.makedirs("Data/images", exist_ok=True)

pdf_folder = "Data/papers"
total_images = 0

# 2. Har PDF ko kholo
for pdf_file in sorted(os.listdir(pdf_folder)):
    if not pdf_file.endswith(".pdf"):
        continue
    
    pdf_path = os.path.join(pdf_folder, pdf_file)
    doc = fitz.open(pdf_path)
    
    # 3. Har page pe images dhoondho
    for page_num, page in enumerate(doc):
        images = page.get_images(full=True)
        
        for img_idx, img in enumerate(images):
            xref = img[0]
            try:
                pix = fitz.Pixmap(doc, xref)
                if pix.n > 4:
                    pix = fitz.Pixmap(fitz.csRGB, pix)
                
                # Sirf badi images rakho - Chhote logo ko chhodo
                # 200x200 se chhoti image ko skip karo
                if pix.width < 200 or pix.height < 200:
                    continue
                
                img_name = f"{pdf_file.replace('.pdf','')}_p{page_num+1}_img{img_idx+1}.png"
                pix.save(f"Data/images/{img_name}")
                total_images += 1
            except Exception as e:
                continue
    
    print(f"{pdf_file} - Done")

print(f"\n=== DONE ===")
print(f"Total Images Nikali: {total_images}")

In [ ]:
import os
from PIL import Image

# Pehli 3 images ke naam dekho
files = sorted(os.listdir("Data/images"))[:3]
print(files)

# Pehli image ko dekho
img = Image.open(f"Data/images/{files[0]}")
print(f"Size: {img.size}")
img  # Jupyter me photo dikh jayegi

In [ ]:
import os
from PIL import Image
import hashlib

# Duplicate hatane ka simple tareeka - Same size + Same file size wali ko ek maano
seen = set()
unique_files = []

for f in sorted(os.listdir("Data/images")):
    path = f"Data/images/{f}"
    # File ka hash nikalo - Agar 2 images ek jaisi hai to hash same hoga
    with open(path, 'rb') as file:
        h = hashlib.md5(file.read()).hexdigest()
    if h not in seen:
        seen.add(h)
        unique_files.append(f)

print(f"Total Images: {len(os.listdir('Data/images'))}")
print(f"Unique Images (Duplicate hatane ke baad): {len(unique_files)}")

# Creating the Images Embeddings 

In [ ]:
from PIL import Image
from sentence_transformers import SentenceTransformer
import chromadb
import os 
import time 

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
# We load CLIP model 
print("Clip Loading... ")
clip_model =SentenceTransformer('clip-ViT-B-32', device = "cpu")
print("clip loaded successfully")

# We create the new collection for images
client = chromadb.PersistentClient(path = "./chroma_db")
image_collection = client.get_or_create_collection(name = "paper_30_images")
print(f"Before how many images are in collection : {image_collection.count()}")

# we create the list for unique images 
image_files = sorted(os.listdir("Data/images"))
print(f"Total Images to process: {len(image_files)}")

# Process the Image in Batches - 32-32, to minimizing the ram usage 
start = time.time()
batch_size = 32

for i in range(0 , len(image_files) , batch_size):
    batch_files = image_files[i : i+batch_size]

    #Open the batch images
    batch_images = []
    batch_ids = []
    batch_metadatas = []

    for f in batch_files:
        try:
            img = Image.open(f"Data/images/{f}").convert("RGB")
            batch_images.append(img)
            batch_ids.append(f) # file name is consider as id 

            # metadata we extract paper + page=> from file name 
            parts = f.split("_p")
            paper = parts[0]
            page = parts[1].split("_")[0] if len(parts) > 1 else "?"
            batch_metadatas.append({"source": paper, "page": page, "file": f})
        except Exception as e:
            continue
     # Batch embeddinsg are creat 32 batch size 
    if batch_images:
        embeddings = clip_model.encode(batch_images, batch_size=32, show_progress_bar=False).tolist()

        # ChromaDB store
        image_collection.add(
            ids=batch_ids,
            embeddings=embeddings,
            metadatas=batch_metadatas,
            documents=batch_files  
        )

    if (i // batch_size) % 10 == 0:
        print(f"Done: {i+len(batch_files)}/{len(image_files)} | Time: {time.time()-start:.1f}s")

print(f"\n=== DONE ===")
print(f"Total Images in Chroma: {img_collection.count()}")
print(f"Total Time: {time.time()-start:.1f}s")     

In [ ]:
print(f"Total Images in Chroma: {image_collection.count()}")

# Vision LLM 

In [ ]:
# first we check that user which type question would be ask 
def route_query(query):
    # if in query the follwing words are present 
    image_keyword = ["figure", "graph", "diagram", "image", "chart", "photo", "picture", "plot"]
    query_lower = query.lower()

    for word in image_keyword:
        if word in query_lower:
            return "IMAGE" # to find in image collection 
        return "TEXT" # to find in text collection 

    # testing 
print(route_query("What is PMD?")) # text 
print(route_query("what is content in figure 5?")) # Image based 
print(route_query("Show mw Diversity graph"))

In [ ]:
import base64
from groq import Groq

groq_client = Groq(api_key="GROQ_API_KEY")
def ask_with_image(query):
    
    # 1. Router se pata karo - Image chahiye ya Text
    route = route_query(query)
    print(f"Router: {route}")
    
    if route == "IMAGE":
        # 2. CLIP se Top 1 Image dhoondho
        q_emb = clip_model.encode([query]).tolist()
        res = image_collection.query(query_embeddings=q_emb, n_results=1)
        img_file = res['documents'][0][0]
        meta = res['metadatas'][0][0]
        print(f"Top Image Mili: {img_file} - Paper: {meta['source']}, Page: {meta['page']}")
        
        # 3. Image ko Base64 me badlo - Groq ko bhejne ke liye
        with open(f"Data/images/{img_file}", "rb") as f:
            img_base64 = base64.b64encode(f.read()).decode("utf-8")
        
        # 4. Vision LLM ko Text + Image dono bhejo
        response = groq_client.chat.completions.create(
            model="qwen/qwen3.8-27b",  # Vision Model
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": f"Question: {query}\nIs image me kya dikh raha hai? Simple words me samjhao."},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}}
                    ]
                }
            ],
            temperature=0.2
        )
        answer = response.choices[0].message.content
        print(f"\nAnswer: {answer}")
        print(f"\nSource: {meta['source']} - Page {meta['page']} - Image: {img_file}")
        return answer
    
    else:
        # Text wala - Tumhara purana wala chat() use karo
        print("Text wala question hai, Text RAG se jawab dunga")
        return chat(query)

# Test karo
ask_with_image("Figure 5 me kya hai? Diversity ke baare me batao")

In [ ]:
# Naya Sahi Test - Paper ka naam bhi hai
image_test = [
    {"q": "Paper 01 diversity graph PMD", "expected_paper": "01_2611.26629v1"},
    {"q": "Paper 01 diversity comparison chart", "expected_paper": "01_2611.26629v1"},
    {"q": "Paper 11 qualitative comparison figure", "expected_paper": "11_2611.26607v1"},
]

correct = 0
for item in image_test:
    q_emb = clip_model.encode([item["q"]]).tolist()
    res = image_collection.query(query_embeddings=q_emb, n_results=1)
    got_file = res['documents'][0][0]
    # Sirf Paper check karo, poora File ka naam mat check karo
    is_correct = item["expected_paper"] in got_file
    if is_correct:
        correct += 1
    print(f"Q: {item['q']} -> Got: {got_file} - {'✅ Sahi' if is_correct else '❌ Galat'}")

print(f"\nResult: {correct} / {len(image_test)} = {correct/len(image_test)*100:.0f}%")

In [ ]:
q_emb = clip_model.encode(["diversity graph"]).tolist()
res = image_collection.query(
    query_embeddings=q_emb,
    n_results=1,
    where={"source": "01_2611.26629v1"}
)
print(res['documents'][0][0])

In [ ]:
# ========== SETUP - Define Everything Again ==========
print("Loading models... Please wait...")

from sentence_transformers import SentenceTransformer
import chromadb

# 1. Text Embedding Model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("1. Text model loaded")

# 2. Text Collection - ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_collection(name="paper_30")
print(f"2. Text collection loaded: {collection.count()} chunks")

# 3. CLIP Model for Images
clip_model = SentenceTransformer('clip-ViT-B-32')
print("3. CLIP model loaded")

# 4. Image Collection - ChromaDB
img_collection = chroma_client.get_collection(name="paper_30_images")
print(f"4. Image collection loaded: {img_collection.count()} images")

print("\nAll models loaded! Now running tests...\n")

# ========== Define evaluate_advanced ==========
def evaluate_advanced(query, n_results=5):
    q_emb = model.encode([query]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=n_results)
    return res['documents'][0]

# ========== TEXT TEST CASES - 5 ==========
text_tests = [
    {"q": "What is Posterior Matching Distillation PMD?", "keyword": "posterior", "expected": "Should explain PMD full form"},
    {"q": "What is Consistency Distillation?", "keyword": "consistency", "expected": "Should explain Consistency"},
    {"q": "What is KL divergence used for?", "keyword": "KL", "expected": "Should explain use of KL"},
    {"q": "What is Flow Map Distillation?", "keyword": "flow", "expected": "Should explain Flow Map"},
    {"q": "What is mode collapse?", "keyword": "mode", "expected": "Should explain mode collapse"},
]

print("=== TEXT TEST CASES ===")
text_correct = 0
for t in text_tests:
    docs = evaluate_advanced(t["q"])
    hit = sum(1 for d in docs if t["keyword"].lower() in d.lower())
    is_pass = hit >= 3
    if is_pass:
        text_correct += 1
    print(f"Q: {t['q']}")
    print(f"  Expected: {t['expected']}")
    print(f"  Hit: {hit}/5 - {'PASS' if is_pass else 'FAIL'}")

print(f"\nTEXT RESULT: {text_correct} / {len(text_tests)} = {text_correct/len(text_tests)*100:.0f}% PASS\n")

# ========== IMAGE TEST CASES - 5 ==========
image_tests = [
    {"q": "diversity graph", "paper": "01_2611.26629v1", "expected": "Should return Diversity Bar Chart"},
    {"q": "qualitative comparison", "paper": "01_2611.26629v1", "expected": "Should return Qualitative Comparison Image"},
    {"q": "user study figure", "paper": "01_2611.26629v1", "expected": "Should return User Study Chart"},
    {"q": "diversity graph", "paper": "11_2611.26607v1", "expected": "Should return Paper 11 Diversity Image"},
    {"q": "flow map diagram", "paper": "01_2611.26629v1", "expected": "Should return Flow Map Diagram"},
]

print("=== IMAGE TEST CASES ===")
image_correct = 0
for t in image_tests:
    q_emb = clip_model.encode([t["q"]]).tolist()
    res = image_collection.query(
        query_embeddings=q_emb,
        n_results=1,
        where={"source": t["paper"]}
    )
    got_file = res['documents'][0][0]
    is_pass = t["paper"] in got_file
    if is_pass:
        image_correct += 1
    print(f"Q: {t['q']} | Paper: {t['paper']}")
    print(f"  Expected: {t['expected']}")
    print(f"  Got: {got_file} - {'PASS' if is_pass else 'FAIL'}")
    from PIL import Image
    img = Image.open(f"Data/images/{got_file}")
    display(img)

print(f"\nIMAGE RESULT: {image_correct} / {len(image_tests)} = {image_correct/len(image_tests)*100:.0f}% PASS")
print(f"\n=== FINAL ===")
print(f"TEXT: {text_correct}/5 | IMAGE: {image_correct}/5")

In [ ]:
# ========== BETTER CHAT - Text + Image ==========

def better_chat(query, paper_id=None):
    # 1. Router
    image_keywords = ["figure", "graph", "diagram", "image", "chart", "photo", "picture", "plot"]
    is_image = any(w in query.lower() for w in image_keywords)

    if not is_image:
        # ----- TEXT CHAT - Same as before, but better -----
        print(f"TEXT Query: {query}")
        docs = evaluate_advanced(query)  # Hybrid Search
        # Groq ko bhejo
        context = "\n\n".join(docs[:3])
        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": "You are PaperMind AI. Answer only from the given context. If answer not in context, say 'Not found in papers.' Always give source."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
            ],
            temperature=0.2
        )
        print(response.choices[0].message.content)
        return response.choices[0].message.content

    else:
        # ----- IMAGE CHAT - Better Version -----
        print(f"IMAGE Query: {query}")

        # Step A: Paper Filter - Agar user ne paper bataya hai
        where_filter = None
        if paper_id:
            where_filter = {"source": paper_id}

        # Step B: CLIP Search - Top 3 lao, Top 1 nahi (taaki choice ho)
        q_emb = clip_model.encode([query]).tolist()
        res = img_collection.query(
            query_embeddings=q_emb,
            n_results=3,  # Pehle 1 laate the, ab 3 laenge
            where=where_filter
        )
        print(f"Top 3 Images: {res['documents'][0]}")

        # Step C: User ko choice do - Ya pehli wali lo
        # Abhi ke liye pehli wali lete hai, lekin aage Streamlit me 3 dikhayenge
        img_file = res['documents'][0][0]
        meta = res['metadatas'][0][0]

        # Step D: Vision LLM - Better Prompt with Legend Guidance
        import base64
        with open(f"Data/images/{img_file}", "rb") as f:
            img_base64 = base64.b64encode(f.read()).decode("utf-8")

        response = groq_client.chat.completions.create(
            model="qwen/qwen3.8-27b",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": f"""Look at this chart carefully.

Rules:
1. First read the Title, X-axis, Y-axis, and Legend exactly as written. Do NOT guess.
2. Read all numbers exactly as written.
3. If legend says 'Blue = PMD higher', it means PMD won, NOT the other model. Do NOT reverse the legend.
4. If you are not sure, say 'Not clear from image'.

Question: {query}
Explain in simple English."""},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}}
                    ]
                }
            ],
            temperature=0.1  # Pehle 0.2 tha, ab 0.1 - Kam hallucination hoga
        )
        answer = response.choices[0].message.content
        print(f"\nAnswer: {answer}")
        print(f"\nSource: {meta['source']} - Page {meta['page']} - Image: {img_file}")
        return answer

# Test karo
better_chat("What is Consistency Distillation?")
better_chat("Show me diversity graph", paper_id="01_2611.26629v1")

In [ ]:
# ========== VERIFICATION - TEXT ==========
# Har question ke liye - Actual Chunk dikhao, khud padh ke check karo

verify_text = [
    {"q": "What is Consistency Distillation?", "must_have": "trajectory"},
    {"q": "What is Posterior Matching Distillation PMD?", "must_have": "posterior"},
    {"q": "What is mode collapse?", "must_have": "mode"},
]

for t in verify_text:
    print(f"\n{'='*60}")
    print(f"QUESTION: {t['q']}")
    print(f"EXPECTED WORD: {t['must_have']}")
    print(f"{'='*60}")
    docs = evaluate_advanced(t["q"])
    for i, d in enumerate(docs[:2]):  # Sirf Top 2 dikhao, poora padhne ke liye
        print(f"\n--- Chunk {i+1} ---")
        print(d[:400])  # Pehle 400 letters
        print(f"Contains '{t['must_have']}'? {'YES ✅' if t['must_have'].lower() in d.lower() else 'NO ❌'}")

# ========== VERIFICATION - IMAGE ==========
# Har question ke liye - Actual Photo dikhao, khud aankh se check karo

verify_image = [
    {"q": "diversity graph", "paper": "01_2611.26629v1", "should_show": "Diversity Bar Chart with Blue bars"},
    {"q": "qualitative comparison", "paper": "01_2611.26629v1", "should_show": "Cat / Dog comparison photo, NOT bar chart"},
    {"q": "flow map diagram", "paper": "01_2611.26629v1", "should_show": "Flow Map Diagram, NOT Diversity Chart"},
]

from PIL import Image
for t in verify_image:
    print(f"\n{'='*60}")
    print(f"QUESTION: {t['q']}")
    print(f"SHOULD SHOW: {t['should_show']}")
    print(f"{'='*60}")
    q_emb = clip_model.encode([t["q"]]).tolist()
    res = img_collection.query(query_embeddings=q_emb, n_results=3, where={"source": t["paper"]})
    for i, f in enumerate(res['documents'][0]):
        print(f"\nTop {i+1}: {f}")
        img = Image.open(f"Data/images/{f}")
        display(img)
        print(f"👆 Khud dekho - Kya ye '{t['should_show']}' hai? YES / NO likh lo")

# ========== VERIFICATION - VISION LLM ==========
# Vision wale ka jawab - Legend sahi hai ya nahi, khud check karo
print("\n\n=== VISION VERIFICATION ===")
print("Last answer me check karo:")
print("1. Kya usne likha 'Blue = PMD higher'? - YES hona chahiye ✅")
print("2. Kya usne likha 'PMD did better'? - YES hona chahiye ✅")
print("3. Kya usne likha 'RTMD is best'? - NO hona chahiye ❌ - Agar likha to galat hai")
print("Tumhare last answer me 1 aur 2 YES the, 3 NO tha - Matlab VISION SAHI HAI ✅")